In [ ]:
import os
import gc
import time
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel
from gensim.models import Word2Vec
import datasets
from numba import njit, prange

# ==========================================
# 0. CONFIGURATION
# ==========================================
class Config:
    # Chemins
    DATA_DIR = "/kaggle/input/data-ctr"
        
    ITEM_INFO_PATH = "/kaggle/input/data-ctr/item_info.parquet"
    SEQ_PATH = "/kaggle/input/data-ctr/item_seq.parquet"
    IMAGE_DIR = "/kaggle/input/data-ctr/item_images/item_images"
    
    # Modèle
    MODEL_ID = "openai/clip-vit-base-patch32"
    INPUT_DIM = 512
    TARGET_DIM = 128
    
    # Entraînement Projecteur
    BATCH_SIZE = 128
    NUM_WORKERS = 2
    PROJ_LR = 1e-3
    PROJ_EPOCHS = 10
    
    # Word2Vec
    LIMIT_W2V_DATA = False 
    
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    OUTPUT_PATH = "item_emb_projector.parquet"
    OUTPUT_INFO_UPDATED = "item_info_updated.parquet"

device = Config.DEVICE
print(f"Running on {device}")

# ==========================================
# 1. HELPER CLASSES (Optimized)
# ==========================================

# --- NUMBA ALIGNMENT KERNEL ---
@njit(parallel=True)
def numba_fast_alignment(clip_ids, w2v_ids_sorted, w2v_vecs_sorted, dim):
    n = len(clip_ids)
    
    # Output arrays
    # mask: 1 if found, 0 if not
    mask = np.zeros(n, dtype=np.bool_)
    # targets: vectors aligned with clip_ids
    targets = np.zeros((n, dim), dtype=np.float32)
    
    # Parallel Loop
    for i in prange(n):
        target_id = clip_ids[i]
        
        # Binary search manually or use np.searchsorted logic
        # Since we are inside Numba, np.searchsorted is supported and fast
        idx = np.searchsorted(w2v_ids_sorted, target_id)
        
        # Check if found
        if idx < len(w2v_ids_sorted) and w2v_ids_sorted[idx] == target_id:
            mask[i] = True
            targets[i] = w2v_vecs_sorted[idx]
            
    return mask, targets

# --- GENERATOR WITH TQDM ---
class SequenceGenerator:
    def __init__(self, sequences, total_len=None):
        self.sequences = sequences
        self.total_len = total_len
        
    def __iter__(self):
        # On wrap l'itérateur avec tqdm pour voir la vitesse de lecture
        iterator = self.sequences
        if self.total_len:
            iterator = tqdm(self.sequences, total=self.total_len, desc="Stream W2V Seq")
            
        for seq in iterator:
            if seq is not None:
                yield [str(x) for x in seq]

# --- CLIP & PROJECTOR ---
class CLIPWrapper(nn.Module):
    def __init__(self, model_id):
        super().__init__()
        self.model = CLIPModel.from_pretrained(model_id)
        self.model.vision_model.requires_grad_(False)
        
    def forward(self, pixel_values):
        vision_outputs = self.model.vision_model(pixel_values=pixel_values)
        image_embeds = self.model.visual_projection(vision_outputs[1])
        return image_embeds / image_embeds.norm(dim=-1, keepdim=True)

class VisualProjector(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2), 
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
    def forward(self, x):
        return self.net(x)

class ImageDataset(Dataset):
    def __init__(self, hf_dataset, image_dir, processor):
        self.data = hf_dataset
        self.image_dir = image_dir
        self.processor = processor
        self.default_img = Image.new("RGB", (224, 224), (0, 0, 0))
        
    def __len__(self): return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        item_id = str(item['item_id'])
        img_path = os.path.join(self.image_dir, f"{item_id}.jpg")
        try: image = Image.open(img_path).convert("RGB")
        except: image = self.default_img
        inputs = self.processor(images=image, return_tensors="pt")
        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "item_id": int(item_id)
        }

# ==========================================
# 2. MAIN PIPELINE
# ==========================================

def main():
    # --- PHASE 1: IMAGES (TEACHER 1) ---
    print("\n" + "="*50)
    print("PHASE 1: CLIP Feature Extraction")
    print("="*50)
    
    hf_ds = datasets.load_dataset("parquet", data_files=Config.ITEM_INFO_PATH, split="train")
    processor = CLIPProcessor.from_pretrained(Config.MODEL_ID)
    clip_model = CLIPWrapper(Config.MODEL_ID).to(device)
    if torch.cuda.device_count() > 1: clip_model = nn.DataParallel(clip_model)

    loader = DataLoader(
        ImageDataset(hf_ds, Config.IMAGE_DIR, processor),
        batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True
    )

    image_features_list = []
    item_ids_list = []

    # TQDM sur l'extraction
    with torch.no_grad():
        for batch in tqdm(loader, desc="[1/4] Extracting CLIP"):
            pixel_values = batch["pixel_values"].to(device)
            ids = batch["item_id"].numpy()
            img_emb = clip_model(pixel_values)
            image_features_list.append(img_emb.cpu().numpy())
            item_ids_list.append(ids)

    raw_image_feats = np.vstack(image_features_list)
    all_item_ids = np.concatenate(item_ids_list)
    
    del clip_model, loader
    torch.cuda.empty_cache()
    gc.collect()

    # --- PHASE 2a: WORD2VEC (TEACHER 2) ---
    print("\n" + "="*50)
    print("PHASE 2a: Behavioral Learning (Word2Vec)")
    print("="*50)

    df_seq = pd.read_parquet(Config.SEQ_PATH, columns=['item_seq'])
    if Config.LIMIT_W2V_DATA:
        seq_data = df_seq['item_seq'].iloc[:1000000]
    else:
        seq_data = df_seq['item_seq']

    # Generator with TQDM
    sentences_stream = SequenceGenerator(seq_data, total_len=len(seq_data))
    
    print("[2/4] Training Word2Vec...")
    w2v_model = Word2Vec(
        sentences=sentences_stream, 
        vector_size=Config.TARGET_DIM, 
        window=5, 
        min_count=1, 
        workers=4,
        epochs=3
    )
    
    # --- NUMBA ALIGNMENT ---
    print("\n[INFO] Preparing Numba Alignment...")
    # 1. Extract Vocab to Arrays
    vocab_keys = list(w2v_model.wv.index_to_key)
    vocab_ids = np.array([int(k) for k in vocab_keys], dtype=np.int64)
    vocab_vecs = np.array([w2v_model.wv[k] for k in vocab_keys], dtype=np.float32)
    
    # 2. Sort for Binary Search (Required for Numba speed)
    print("[INFO] Sorting W2V Vocab...")
    sort_idx = np.argsort(vocab_ids)
    vocab_ids_sorted = vocab_ids[sort_idx]
    vocab_vecs_sorted = vocab_vecs[sort_idx]
    
    # 3. Run Numba Kernel
    print("[INFO] Running Numba Fast Alignment...")
    mask, targets = numba_fast_alignment(
        all_item_ids.astype(np.int64), 
        vocab_ids_sorted, 
        vocab_vecs_sorted, 
        Config.TARGET_DIM
    )
    
    valid_count = np.sum(mask)
    print(f"✅ Items Aligned: {valid_count} / {len(all_item_ids)} ({valid_count/len(all_item_ids)*100:.1f}%)")
    
    # Filter Data for Training
    X_train = raw_image_feats[mask]
    y_train = targets[mask]
    
    del df_seq, sentences_stream, w2v_model, vocab_ids, vocab_vecs
    gc.collect()

    # --- PHASE 2b: PROJECTOR TRAINING ---
    print("\n" + "="*50)
    print("PHASE 2b: Projector Training (Distillation)")
    print("="*50)
    
    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

    projector = VisualProjector(Config.INPUT_DIM, Config.TARGET_DIM).to(device)
    opt = optim.Adam(projector.parameters(), lr=Config.PROJ_LR)
    crit = nn.MSELoss()
    
    projector.train()
    
    # Loop with TQDM
    for ep in range(Config.PROJ_EPOCHS):
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Ep {ep+1}/{Config.PROJ_EPOCHS}", leave=True)
        
        for bx, by in pbar:
            bx, by = bx.to(device), by.to(device)
            
            opt.zero_grad()
            pred = projector(bx)
            loss = crit(pred, by)
            loss.backward()
            opt.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    # --- PHASE 3: INFERENCE ---
    print("\n" + "="*50)
    print("PHASE 3: Final Inference & Save")
    print("="*50)
    
    projector.eval()
    final_embs_list = []
    
    # Infer on ALL items (Cold Start solution)
    inf_loader = DataLoader(TensorDataset(torch.FloatTensor(raw_image_feats)), batch_size=1024, shuffle=False)
    
    with torch.no_grad():
        for (bx,) in tqdm(inf_loader, desc="[4/4] Projecting"):
            bx = bx.to(device)
            emb = projector(bx)
            # L2 Normalize
            emb = emb / (emb.norm(dim=-1, keepdim=True) + 1e-6)
            final_embs_list.append(emb.cpu().numpy())

    final_embs = np.vstack(final_embs_list)

    # Save
    print(f"Saving to {Config.OUTPUT_PATH}...")
    df_out = pl.DataFrame({
        "item_id": all_item_ids,
        "item_emb": list(final_embs)
    })
    df_out.write_parquet(Config.OUTPUT_PATH)
    
    # Update item_info
    print("Updating item_info.parquet...")
    df_info = pl.read_parquet(Config.ITEM_INFO_PATH)
    if "item_emb" in df_info.columns:
        df_info = df_info.drop("item_emb")
    
    df_info = df_info.join(df_out, on="item_id", how="left")
    df_info.write_parquet(Config.OUTPUT_INFO_UPDATED)
    
    print("\n✅ [DONE] Projector Pipeline Completed.")

if __name__ == "__main__":
    main()

In [ ]:
a = pl.read_parquet("/kaggle/working/item_info_updated.parquet")
a

In [ ]:
import os
import shutil
import glob
import sys
import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm
from numba import njit, prange

# ==========================================
# CONFIGURATION
# ==========================================
class Config:
    # 1. Repo Settings
    REPO_URL = "https://github.com/Othmane999/WWW2025_MMCTR_Challenge"
    REPO_DIR = "WWW2025_MMCTR_Challenge"
    
    # 2. Data Paths (Input)
    SOURCE_DATA_DIR = "/kaggle/input/data-ctr"
    
    # This is the output from your Part 1
    MY_UPDATED_INFO = "item_info_updated.parquet" 
    
    # 3. FuxiCTR Internal Paths
    DATASET_ID = "MicroLens_1M_x1"
    DEST_DATA_DIR = os.path.join(REPO_DIR, "data", DATASET_ID)
    
    # 4. Outputs
    WORKING_DIR = "/kaggle/working"
    FINAL_SUBMISSION = os.path.join(WORKING_DIR, "prediction.csv")
    BEST_MODEL_DEST = os.path.join(WORKING_DIR, "best_model.pth")

# ==========================================
# 0. NUMBA KERNEL (Data Integrity)
# ==========================================
@njit(parallel=True, fastmath=True)
def check_embedding_integrity(emb_matrix):
    """
    Scans the entire embedding matrix for NaNs or Infs.
    """
    rows, cols = emb_matrix.shape
    has_error = False
    
    for i in prange(rows):
        if has_error: break 
        for j in range(cols):
            val = emb_matrix[i, j]
            if np.isnan(val) or np.isinf(val):
                has_error = True
                
    return not has_error 

# ==========================================
# 1. SETUP & DATA MIGRATION (WITH FIX)
# ==========================================
def setup_environment_and_data():
    print("\n" + "="*40)
    print("=== Phase 1: Environment & Data Setup ===")
    print("="*40)
    
    # A. Clone Repo
    if not os.path.exists(Config.REPO_DIR):
        print(f"[INFO] Cloning {Config.REPO_URL}...")
        os.system(f"git clone {Config.REPO_URL} {Config.REPO_DIR}")
    else:
        print(f"[INFO] Repo {Config.REPO_DIR} already exists.")

    # B. Install Requirements
    print("[INFO] Installing requirements...")
    os.system(f"pip install -q -r {Config.REPO_DIR}/requirements.txt")
    
    # C. Create Data Directory
    os.makedirs(Config.DEST_DATA_DIR, exist_ok=True)
    
    # D. Copy Standard Files with TQDM
    files_to_copy = ["train.parquet", "valid.parquet", "test.parquet", "item_seq.parquet"]
    
    print(f"[INFO] Migrating Base Data to {Config.DEST_DATA_DIR}...")
    for fname in tqdm(files_to_copy, desc="Copying Datasets"):
        src = os.path.join(Config.SOURCE_DATA_DIR, fname)
        dst = os.path.join(Config.DEST_DATA_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, dst)
        else:
            print(f"⚠️ WARNING: Missing source file {src}")

    # =========================================================
    # E. CRITICAL FIX: RENAME & INJECT EMBEDDINGS
    # =========================================================
    print(f"[INFO] Processing Custom Embeddings from {Config.MY_UPDATED_INFO}...")
    
    if os.path.exists(Config.MY_UPDATED_INFO):
        # 1. Load Data
        df = pl.read_parquet(Config.MY_UPDATED_INFO)
        print(f"   Original Columns: {df.columns}")
        
        # 2. FIX: Handle Column Names
        # We need 'item_emb_d128'.
        # If Part 1 created 'item_emb', we must rename it.
        # If Part 1 left the old 'item_emb_d128', we must drop it first.
        
        # Step 2a: Drop the OLD original embedding if it exists alongside our new one
        if "item_emb" in df.columns and "item_emb_d128" in df.columns:
            print("   [FIX] Dropping old 'item_emb_d128' to replace with new 'item_emb'...")
            df = df.drop("item_emb_d128")
            
        # Step 2b: Rename 'item_emb' -> 'item_emb_d128'
        if "item_emb" in df.columns:
            print("   [FIX] Renaming 'item_emb' -> 'item_emb_d128'...")
            df = df.rename({"item_emb": "item_emb_d128"})
            
        # Step 2c: Verify the target column exists now
        target_col = "item_emb_d128"
        if target_col not in df.columns:
            print(f"🔴 ERROR: Could not find or create '{target_col}'. Available: {df.columns}")
            sys.exit(1)

        # 3. Integrity Check (Numba)
        print("[INFO] Running Numba Integrity Check (NaN/Inf scan)...")
        emb_list = df[target_col].to_list()
        emb_matrix = np.array(emb_list, dtype=np.float32)
        
        is_valid = check_embedding_integrity(emb_matrix)
        
        if not is_valid:
            print("🔴 CRITICAL ERROR: Your embeddings contain NaNs or Infinity values.")
            sys.exit(1)
        else:
            print(f"✅ Integrity Passed: {emb_matrix.shape} matrix is clean.")

        # 4. Save to Repo Directory
        dst_info = os.path.join(Config.DEST_DATA_DIR, "item_info.parquet")
        print(f"   Saving corrected file to: {dst_info}")
        df.write_parquet(dst_info)
        print("✅ Custom item_info.parquet injected successfully!")
        
    else:
        print(f"🔴 ERROR: {Config.MY_UPDATED_INFO} not found! Did you run Part 1?")
        sys.exit(1)

# ==========================================
# 2. RUN OFFICIAL BASELINE
# ==========================================
def run_official_training():
    print("\n" + "="*40)
    print("=== Phase 2: Running FuxiCTR Baseline ===")
    print("="*40)
    
    original_cwd = os.getcwd()
    os.chdir(Config.REPO_DIR)
    
    # Run Tuner (Full Training, no epoch limit)
    cmd = "python run_param_tuner.py --config config/DIN_microlens_mmctr_tuner_config_01.yaml --gpu 0"
    print(f"[EXEC] {cmd}")
    
    exit_code = os.system(cmd)
    
    os.chdir(original_cwd)
    
    if exit_code != 0:
        raise RuntimeError(f"Training failed with exit code {exit_code}")
    print("✅ Training Completed Successfully.")

# ==========================================
# 3. EXPORT RESULTS
# ==========================================
def export_submission():
    print("\n" + "="*40)
    print("=== Phase 3: Formatting & Exporting ===")
    print("="*40)
    
    exp_dir = os.path.join(Config.REPO_DIR, "model_experiment")
    
    # 1. Find CSV
    csv_files = glob.glob(os.path.join(exp_dir, "**", "*.csv"), recursive=True)
    if not csv_files:
        print("🔴 No result CSV found.")
        return

    latest_csv = max(csv_files, key=os.path.getmtime)
    print(f"[INFO] Processing Raw Output: {os.path.basename(latest_csv)}")
    
    try:
        df = pd.read_csv(latest_csv)
        
        # 2. Rename Logic
        rename_map = {}
        for col in df.columns:
            if 'id' in col.lower(): rename_map[col] = 'ID'
            elif 'pred' in col.lower() or 'score' in col.lower(): rename_map[col] = 'Task1'
            
        df = df.rename(columns=rename_map)
        
        # 3. Validate & Save
        if 'ID' in df.columns and 'Task1' in df.columns:
            final_df = df[['ID', 'Task1']].sort_values('ID')
            final_df.to_csv(Config.FINAL_SUBMISSION, index=False)
            print(f"✅ Submission Ready: {Config.FINAL_SUBMISSION}")
            print("   Preview:")
            print(final_df.head())
        else:
            print(f"🔴 Column mismatch. Found: {df.columns}")

    except Exception as e:
        print(f"🔴 Error processing CSV: {e}")

    # 4. Save Model
    pth_files = glob.glob(os.path.join(exp_dir, "**", "*.pth"), recursive=True)
    if pth_files:
        latest_pth = max(pth_files, key=os.path.getmtime)
        shutil.copy(latest_pth, Config.BEST_MODEL_DEST)
        print(f"✅ Best Model Saved: {Config.BEST_MODEL_DEST}")

# ==========================================
# MAIN
# ==========================================
if __name__ == "__main__":
    setup_environment_and_data()
    run_official_training()
    export_submission()

In [ ]:
import os
import shutil
import glob

def save_best_model_absolute():
    print("\n" + "="*40)
    print("=== Saving Best Model (Robust Mode) ===")
    print("="*40)

    # 1. Use ABSOLUTE paths to avoid directory errors
    # This path is fixed and does not depend on os.getcwd()
    BASE_PATH = "/kaggle/working/WWW2025_MMCTR_Challenge/checkpoints/MicroLens_1M_x1"
    DEST_PATH = "/kaggle/working/best_model.pth"

    print(f"[INFO] Searching in: {BASE_PATH}")

    # 2. Check if directory exists
    if not os.path.exists(BASE_PATH):
        print(f"🔴 Error: Directory not found at {BASE_PATH}")
        # Debug: List what IS in working
        print("Contents of /kaggle/working/:", os.listdir("/kaggle/working"))
        return

    # 3. Find .model files
    pattern = os.path.join(BASE_PATH, "*.model")
    model_files = glob.glob(pattern)

    if not model_files:
        print("🔴 Error: No .model files found in directory.")
        print("Files present:", os.listdir(BASE_PATH))
        return

    # 4. Pick the newest one
    best_model = max(model_files, key=os.path.getmtime)
    print(f"[INFO] Found Best Model: {os.path.basename(best_model)}")

    # 5. Copy
    try:
        shutil.copy(best_model, DEST_PATH)
        print(f"✅ [SUCCESS] Model copied to: {DEST_PATH}")
    except Exception as e:
        print(f"🔴 Copy failed: {e}")

if __name__ == "__main__":
    save_best_model_absolute()

In [ ]:
exp_dir = os.path.join(Config.REPO_DIR, "model_experiment")
    pth_files = glob.glob(os.path.join(exp_dir, "**", "*.pth"), recursive=True)
    if pth_files:
        best = max(pth_files, key=os.path.getmtime)
        shutil.copy(best, Config.BEST_MODEL_DEST)
        print(f"✅ Best Model Saved: {Config.BEST_MODEL_DEST}")
    else:
        print("⚠️ No model file found after training.")


In [ ]:
import sys
import os
import glob
import pandas as pd
import polars as pl
import torch
import numpy as np
import importlib.util
from tqdm.auto import tqdm

# ==========================================
# 0. CONFIGURATION
# ==========================================
class Config:
    MODEL_PATH = "/kaggle/working/best_model.pth"
    REPO_DIR = "/kaggle/working/WWW2025_MMCTR_Challenge"
    DATASET_ID = "MicroLens_1M_x1"
    
    TEST_DATA_PATH = "/kaggle/input/data-ctr/test.parquet"
    ITEM_INFO_PATH = os.path.join(REPO_DIR, "data", DATASET_ID, "item_info.parquet")
    FEATURE_MAP_PATH = os.path.join(REPO_DIR, "data", DATASET_ID, "feature_map.json")
    DIN_SOURCE_PATH = os.path.join(REPO_DIR, "src", "DIN.py")
    
    OUTPUT_CSV = "/kaggle/working/prediction.csv"
    BATCH_SIZE = 4096

# ==========================================
# 1. SETUP
# ==========================================
def load_din_class(source_path):
    if not os.path.exists(source_path): return None
    try:
        spec = importlib.util.spec_from_file_location("DIN_Custom", source_path)
        module = importlib.util.module_from_spec(spec)
        sys.modules["DIN_Custom"] = module
        spec.loader.exec_module(module)
        return getattr(module, "DIN")
    except: return None

def pad_sequence(col_data, max_len):
    batch_list = col_data.to_list()
    padded_batch = []
    for seq in batch_list:
        if seq is None: seq = []
        else: seq = list(seq)
        if len(seq) > max_len: seq = seq[:max_len]
        elif len(seq) < max_len: seq = seq + [0] * (max_len - len(seq))
        padded_batch.append(seq)
    return torch.tensor(padded_batch, dtype=torch.long).to("cuda")

# ==========================================
# 2. MAIN EXECUTION
# ==========================================
def run_final_inference():
    print("\n" + "="*50)
    print("=== Phase 10: Strict Dimension Match ===")
    print("="*50)

    # A. INITIALIZE
    try: from fuxictr.features import FeatureMap
    except: 
        print("🔴 fuxictr not installed.")
        return

    # Params
    params = {
        "model_id": "DIN_Inference", "dataset_id": Config.DATASET_ID,
        "data_root": os.path.dirname(Config.ITEM_INFO_PATH), "model_root": "/kaggle/working/",
        "feature_cols": [], "dnn_hidden_units": [1024, 512, 256],
        "attention_hidden_units": [512, 256], "dnn_activations": "ReLU",
        "attention_hidden_activations": "ReLU", "embedding_dim": 64,
        "batch_norm": True, "din_use_softmax": False, "gpu": 0,
        "task": "binary_classification", "loss": "binary_crossentropy",
        "metrics": ["AUC", "logloss"], "optimizer": "adam",
        "learning_rate": 0.001, "verbose": 0, "early_stop_patience": 3,
        "attention_dropout": 0.1, "net_dropout": 0.1
    }

    print("[INFO] Loading FeatureMap...")
    feature_map = FeatureMap(Config.DATASET_ID, os.path.dirname(Config.ITEM_INFO_PATH))
    feature_map.load(Config.FEATURE_MAP_PATH, params)

    DIN = load_din_class(Config.DIN_SOURCE_PATH)
    print(f"[INFO] Loading Weights from {Config.MODEL_PATH}...")
    model = DIN(feature_map, **params)
    state_dict = torch.load(Config.MODEL_PATH, map_location="cuda:0")
    model.load_state_dict(state_dict, strict=True)
    model.to("cuda")
    model.eval()

    # B. PREPARE DATA
    print("[INFO] Loading & Joining Data...")
    df_test = pl.read_parquet(Config.TEST_DATA_PATH)
    df_info = pl.read_parquet(Config.ITEM_INFO_PATH)
    df = df_test.join(df_info, on="item_id", how="left")
    print(f"   Data Loaded. Rows: {len(df)}")

    # C. INFERENCE LOOP
    print("[INFO] Starting Prediction...")
    all_preds = []
    
    num_batches = (len(df) + Config.BATCH_SIZE - 1) // Config.BATCH_SIZE
    
    with torch.no_grad():
        for i in tqdm(range(num_batches), desc="Inferring"):
            batch_df = df.slice(i * Config.BATCH_SIZE, Config.BATCH_SIZE)
            current_bs = len(batch_df)
            
            # --- 1. USER/CONTEXT INPUT (Batch Dict) ---
            # STRICTLY exclude item features here to avoid double counting dimensions
            batch_input = {}
            
            # User Features Only
            for col in ['likes_level', 'views_level']: 
                if col in batch_df.columns:
                    batch_input[col] = torch.tensor(
                        batch_df[col].fill_null(0).to_numpy(), dtype=torch.long
                    ).to("cuda")
            
            # User ID is Meta (ignored usually), but safe to add if needed, 
            # as long as it's not source="item".
            if 'user_id' in batch_df.columns:
                 batch_input['user_id'] = torch.tensor(batch_df['user_id'].to_numpy(), dtype=torch.long).to("cuda")


            # --- 2. ITEM INPUT (Simulated History) ---
            # Get Targets
            target_ids = torch.tensor(batch_df['item_id'].to_numpy(), dtype=torch.long).to("cuda")
            
            target_tags = None
            if 'item_tags' in batch_df.columns:
                target_tags = pad_sequence(batch_df['item_tags'], 5)
                
            target_emb = None
            if 'item_emb_d128' in batch_df.columns:
                embs = np.array(batch_df['item_emb_d128'].to_list(), dtype=np.float32)
                target_emb = torch.tensor(embs).to("cuda")
            
            # Create Dummies (Padding/Zeros)
            dummy_ids = torch.zeros_like(target_ids)
            
            item_input = {}
            
            # Stack [Dummy, Target] -> Flatten to [2*B]
            item_input['item_id'] = torch.stack([dummy_ids, target_ids], dim=1).view(-1)
            
            if target_tags is not None:
                dummy_tags = torch.zeros_like(target_tags)
                item_input['item_tags'] = torch.stack([dummy_tags, target_tags], dim=1).view(-1, 5)

            if target_emb is not None:
                dummy_emb = torch.zeros_like(target_emb)
                item_input['item_emb_d128'] = torch.stack([dummy_emb, target_emb], dim=1).view(-1, 128)

            # --- 3. Mask ---
            mask = torch.ones((current_bs, 1)).to("cuda")

            # --- 4. Forward ---
            output = model.forward((batch_input, item_input, mask))
            
            if isinstance(output, dict): pred = output['y_pred']
            else: pred = output
                
            all_preds.extend(pred.cpu().numpy().flatten())

    # D. SAVE
    print(f"[INFO] Saving {len(all_preds)} predictions...")
    submission = pd.DataFrame({
        "ID": range(len(all_preds)),
        "Task1": all_preds
    })
    
    submission.to_csv(Config.OUTPUT_CSV, index=False)
    print(f"\n✅ [SUCCESS] Submission saved to: {Config.OUTPUT_CSV}")
    print(submission.head())

if __name__ == "__main__":
    run_final_inference()